In [1]:
import pandas as pd
import numpy as np
import pandas_ta_classic as ta

In [2]:
# Read the CSV, skipping the two yfinance artifact rows (rows 1 and 2)
df = pd.read_csv('LTTS_yfinance.csv', skiprows=[1, 2])

# Rename the mislabeled 'Price' column to 'Date'
df.rename(columns={'Price': 'Date'}, inplace=True)

# Convert the Date column from text to actual datetime objects
df['Date'] = pd.to_datetime(df['Date'])

# Set Date as the row index (each row = one trading day)
df.set_index('Date', inplace=True)

# Quick look at the data
print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'\nColumn data types:\n{df.dtypes}')
print(f'\nFirst 5 rows:')
df.head()

Dataset shape: 988 rows, 5 columns
Date range: 2022-06-01 to 2026-05-29

Column data types:
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

First 5 rows:


,Close,High,Low,Open,Volume
Date,,,,,
2022-06-01,3301.987549,3354.793656,3266.357745,3345.330204,280493
2022-06-02,3414.271240,3425.580018,3286.656512,3302.034620,360137
2022-06-03,3381.243896,3536.397139,3369.935118,3476.872121,649515
2022-06-06,3301.230225,3376.559389,3227.131587,3373.720353,341348
2022-06-07,3259.685547,3335.866324,3241.610264,3279.085621,243751


In [3]:
# Check if dates are in ascending (oldest-to-newest) order
is_sorted = df.index.is_monotonic_increasing

if is_sorted:
    print('Data is already sorted in ascending chronological order.')
else:
    df.sort_index(inplace=True)
    print('Data was NOT sorted. It has now been sorted ascending by date.')

Data is already sorted in ascending chronological order.


In [4]:
# Detect rows where Open == High == Low == Close AND Volume == 0
placeholder_mask = (
    (df['Open'] == df['High']) &
    (df['High'] == df['Low']) &
    (df['Low'] == df['Close']) &
    (df['Volume'] == 0)
)

# Show which rows were detected
placeholder_rows = df[placeholder_mask]
print(f'Found {len(placeholder_rows)} non-trading placeholder row(s):')
for date in placeholder_rows.index:
    print(f'  - {date.date()}')

# Remove them
df = df[~placeholder_mask]
print(f'\nDataset shape after removal: {df.shape[0]} rows, {df.shape[1]} columns')

Found 4 non-trading placeholder row(s):
  - 2025-03-18
  - 2026-01-15
  - 2026-05-01
  - 2026-05-28

Dataset shape after removal: 984 rows, 5 columns


In [5]:
# Convert all OHLCV columns to numeric (errors='coerce' turns bad values into NaN)
numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check for any NaN values that appeared from failed conversions
nan_counts = df[numeric_columns].isna().sum()
print('NaN counts after numeric conversion (should all be 0):')
print(nan_counts)

NaN counts after numeric conversion (should all be 0):
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


In [6]:
# Create 5 lag features: each is the closing price from N days ago
for lag in range(1, 6):
    df[f'Close_lag_{lag}'] = df['Close'].shift(lag)

print('Lag features created: Close_lag_1 through Close_lag_5')
# Show a few rows so you can see the shifting pattern
lag_cols = ['Close'] + [f'Close_lag_{i}' for i in range(1, 6)]
df[lag_cols].iloc[5:8]

Lag features created: Close_lag_1 through Close_lag_5


,Close,Close_lag_1,Close_lag_2,Close_lag_3,Close_lag_4,Close_lag_5
Date,,,,,,
2022-06-08,3267.019775,3259.685547,3301.230225,3381.243896,3414.271240,3301.987549
2022-06-09,3326.308594,3267.019775,3259.685547,3301.230225,3381.243896,3414.271240
2022-06-10,3261.956787,3326.308594,3267.019775,3259.685547,3301.230225,3381.243896


In [7]:
# Percent change from the previous day's close
df['Daily_Return'] = df['Close'].pct_change()

print('Daily_Return created.')
print(df[['Close', 'Daily_Return']].iloc[1:4])

Daily_Return created.
                  Close  Daily_Return
Date                                 
2022-06-02  3414.271240      0.034005
2022-06-03  3381.243896     -0.009673
2022-06-06  3301.230225     -0.023664


In [8]:
# 20-day Simple Moving Average
df['SMA_20'] = df['Close'].rolling(window=20).mean()

# 20-day Rolling Volatility (standard deviation)
df['Volatility_20'] = df['Close'].rolling(window=20).std()

print('SMA_20 and Volatility_20 created.')

SMA_20 and Volatility_20 created.


In [9]:
# Log-transform volume to compress its wide range
df['Log_Volume'] = np.log1p(df['Volume'])

print('Log_Volume created.')
print(f'Original Volume range: {df["Volume"].min():>12,.0f}  to  {df["Volume"].max():,.0f}')
print(f'Log_Volume range:      {df["Log_Volume"].min():>12.2f}  to  {df["Log_Volume"].max():.2f}')

Log_Volume created.
Original Volume range:        8,405  to  3,433,441
Log_Volume range:              9.04  to  15.05


In [10]:
# 14-day Relative Strength Index
df['RSI_14'] = ta.rsi(df['Close'], length=14)

print('RSI_14 created.')
print(f'RSI range (excluding warm-up NaN): '
      f'{df["RSI_14"].dropna().min():.1f} to {df["RSI_14"].dropna().max():.1f}')

RSI_14 created.
RSI range (excluding warm-up NaN): 17.4 to 82.9


In [11]:
# Compute MACD with standard parameters: 12-day fast, 26-day slow, 9-day signal
macd_result = ta.macd(df['Close'], fast=12, slow=26, signal=9)

# Show what columns pandas_ta generated (for learning purposes)
print(f'MACD result columns: {macd_result.columns.tolist()}')

# Extract the MACD line and signal line
df['MACD'] = macd_result['MACD_12_26_9']           # The main MACD line
df['MACD_Signal'] = macd_result['MACDs_12_26_9']    # The signal line

print('MACD and MACD_Signal created.')

MACD result columns: ['MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9']
MACD and MACD_Signal created.


In [12]:
# 10-day Rate of Change
df['ROC_10'] = ta.roc(df['Close'], length=10)

print('ROC_10 created.')

ROC_10 created.


In [13]:
# Bollinger Bands: 20-day window, 2 standard deviations
bb_result = ta.bbands(df['Close'], length=20, std=2)

# Show what columns pandas_ta generated
print(f'Bollinger Bands columns: {bb_result.columns.tolist()}')

# Extract the three main bands
df['BB_Lower'] = bb_result['BBL_20_2.0']    # Lower band
df['BB_Middle'] = bb_result['BBM_20_2.0']   # Middle band (= SMA_20)
df['BB_Upper'] = bb_result['BBU_20_2.0']    # Upper band

print('Bollinger Bands created (BB_Lower, BB_Middle, BB_Upper).')

Bollinger Bands columns: ['BBL_20_2.0', 'BBM_20_2.0', 'BBU_20_2.0', 'BBB_20_2.0', 'BBP_20_2.0']
Bollinger Bands created (BB_Lower, BB_Middle, BB_Upper).


In [14]:
# On-Balance Volume
df['OBV'] = ta.obv(df['Close'], df['Volume'])

print('OBV created.')

OBV created.


In [15]:
# Day of week as an integer (0=Monday ... 4=Friday, 5=Saturday)
df['Day_of_Week'] = df.index.dayofweek

print('Day_of_Week created.')
print(f'\nDay-of-week distribution:')
print(df['Day_of_Week'].value_counts().sort_index())

Day_of_Week created.

Day-of-week distribution:
Day_of_Week
0    196
1    196
2    196
3    197
4    198
5      1
Name: count, dtype: int64


In [16]:
# Target = next day's closing price (shift Close up by 1 row)
df['Target_Close_Next'] = df['Close'].shift(-1)

print('Target_Close_Next created.')
print(f'Example: On {df.index[0].date()}, Close = {df["Close"].iloc[0]:.2f}, '
      f'Target (next day Close) = {df["Target_Close_Next"].iloc[0]:.2f}')

Target_Close_Next created.
Example: On 2022-06-01, Close = 3301.99, Target (next day Close) = 3414.27


In [17]:
# Calculate the percentage change to tomorrow's close
df['Target_Pct_Change'] = (df['Target_Close_Next'] - df['Close']) / df['Close']

# Define the threshold band
threshold = 0.001  # 0.1%

# Create the label column using numpy.select
conditions = [
    df['Target_Pct_Change'] > threshold,
    df['Target_Pct_Change'] < -threshold
]
choices = ['Up', 'Down']
# Anything not strictly greater than threshold or less than -threshold becomes 'Flat'
df['Target_Class_Next'] = np.select(conditions, choices, default='Flat')

# We also must replace 'Flat' with NaN for the very last row where Target_Close_Next is NaN
df.loc[df['Target_Close_Next'].isna(), 'Target_Class_Next'] = np.nan

print('Target_Class_Next created.')
print('\nClass distribution before dropping NaN:')
print(df['Target_Class_Next'].value_counts())

Target_Class_Next created.

Class distribution before dropping NaN:
Target_Class_Next
Up      474
Down    455
Flat     54
Name: count, dtype: int64


In [18]:
# Show NaN counts per column before dropping
nan_before = df.isna().sum()
print('NaN counts per column (only showing columns that have NaN):')
print(nan_before[nan_before > 0])

rows_before = len(df)

# Drop all rows that have any NaN values
df.dropna(inplace=True)

rows_after = len(df)
print(f'\nRows before dropping NaN: {rows_before}')
print(f'Rows after dropping NaN:  {rows_after}')
print(f'Rows lost to warm-up:     {rows_before - rows_after}')

NaN counts per column (only showing columns that have NaN):
Close_lag_1           1
Close_lag_2           2
Close_lag_3           3
Close_lag_4           4
Close_lag_5           5
Daily_Return          1
SMA_20               19
Volatility_20        19
RSI_14               13
MACD                 25
MACD_Signal          33
ROC_10               10
BB_Lower             19
BB_Middle            19
BB_Upper             19
Target_Close_Next     1
Target_Pct_Change     1
Target_Class_Next     1
dtype: int64

Rows before dropping NaN: 984
Rows after dropping NaN:  950
Rows lost to warm-up:     34


In [19]:
# Save to CSV (the Date index is included automatically)
df.to_csv('cleaned_data.csv')

print('Saved cleaned_data.csv')
print(f'  Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'  Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'\nColumns in the final dataset:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

Saved cleaned_data.csv
  Shape: 950 rows x 26 columns
  Date range: 2022-07-18 to 2026-05-27

Columns in the final dataset:
   1. Close
   2. High
   3. Low
   4. Open
   5. Volume
   6. Close_lag_1
   7. Close_lag_2
   8. Close_lag_3
   9. Close_lag_4
  10. Close_lag_5
  11. Daily_Return
  12. SMA_20
  13. Volatility_20
  14. Log_Volume
  15. RSI_14
  16. MACD
  17. MACD_Signal
  18. ROC_10
  19. BB_Lower
  20. BB_Middle
  21. BB_Upper
  22. OBV
  23. Day_of_Week
  24. Target_Close_Next
  25. Target_Pct_Change
  26. Target_Class_Next
